# Sales Team Performance Analysis

This notebook evaluates CRM sales-team performance across lead volume, conversion, revenue, call activity, and lost-deal reasons.

### Business questions
- Which managers handle the largest share of leads?
- Which managers convert leads most effectively?
- How do managers compare on realized and potential revenue?
- Which managers have the strongest call-success rates?
- What are the most common reasons for lost deals?

> **Limitation:** the dataset does not contain each manager's employment period, so comparisons are based on observed absolute volumes rather than tenure-adjusted performance.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import colors

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.options.display.max_columns = None


## Load Processed Data

In [ ]:
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
contacts = pd.read_pickle(PROCESSED_DIR / 'contacts_clean.pkl')
calls = pd.read_pickle(PROCESSED_DIR / 'calls_clean.pkl')

## 1. Data Preparation

In [ ]:
deals['deal_owner_name'].unique()

In [ ]:
# Managers present in Calls but not in Deals
only_calls = set(calls['call_owner_name']) - set(deals['deal_owner_name'])
print('Managers only in Calls:', only_calls)
print(calls[calls['call_owner_name'].isin(only_calls)]['call_owner_name'].value_counts())

Six managers appear in Calls but not in Deals.

Two of them, **Derek James** and **Fiona Jackson**, have substantial call activity but no deal ownership. They may have operated in call-center or support roles without responsibility for creating or managing deals in the CRM, so they are excluded from deal-owner performance comparisons.

### Analysis Limitation

Managers may have worked for different lengths of time, but employment-period information is not available in the dataset. Therefore, manager comparisons use observed absolute volumes and should not be interpreted as tenure-adjusted productivity.

In [ ]:
mgr_leads = deals.groupby('deal_owner_name')['contact_name'].nunique().rename('total_leads')

mgr_buyers = (deals[deals['is_buyer'] == True].groupby('deal_owner_name')['contact_name'].nunique().rename('buyers'))

mgr_revenue = (deals[deals['is_buyer'] == True].groupby('deal_owner_name')['initial_amount_paid'].sum().rename('revenue'))
mgr_revenue_potentially = (deals[deals['is_buyer'] == True].groupby('deal_owner_name')['offer_total_amount'].sum().rename('revenue_potential'))

sla_99 = deals['sla_minutes'].quantile(0.99)
mgr_sla = (deals[deals['sla_minutes'] <= sla_99].groupby('deal_owner_name')['sla_minutes'].median() / 60).round(1).rename('median_sla_h')

mgr = pd.concat([mgr_leads, mgr_buyers, mgr_revenue, mgr_revenue_potentially, mgr_sla], axis=1).fillna(0).reset_index()

# Realized revenue proxy uses initial_amount_paid.
# Potential revenue uses offer_total_amount (full contract value).

In [ ]:
mgr['conversion'] = (mgr['buyers'] / mgr['total_leads'] * 100).round(2)

In [ ]:
mgr['%_leads'] = (mgr['total_leads'] / mgr['total_leads'].sum() * 100).round(2)

In [ ]:
mgr

In [ ]:
print(mgr[mgr['conversion'] == 0][['deal_owner_name', 'total_leads', 'buyers', 'revenue', 'revenue_potential']])

These are CRM deal owners with recorded leads but no confirmed buyers in the analyzed period.

In [ ]:
mgr_active = mgr[mgr['total_leads'] >= 50].reset_index(drop=True)
print(f'Active managers: {len(mgr_active)}')

## 2. Conversion Rate

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

top_conv = mgr_active.sort_values('conversion', ascending=False)

sns.barplot(data=top_conv, x='conversion', y='deal_owner_name', color=colors['accent'], ax=ax)

# Value labels
ax.bar_label(ax.containers[0], fmt='%.1f', padding=3)

# Titles and axis labels
ax.set_title('Conversion Rate by Manager')
ax.set_xlabel('Conversion (%)')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Managers with leads but no confirmed buyers
print(mgr_active[mgr_active['conversion'] == 0][['deal_owner_name', 'total_leads', 'buyers']])

### Conversion Insights

- **Oliver Taylor** shows a very high conversion rate, but on a relatively small lead base, so the result should be interpreted cautiously.
- **Ulysses Adams, Charlie Davis, Paula Underwood, and Kevin Parker** are among the strongest converters among managers with larger lead volumes.
- Several managers have 50+ leads but no confirmed buyers, suggesting either role differences, short tenure, or performance issues that would require additional operational context.

## 3. Share of Processed Leads

In [ ]:
# Share of observed leads handled by each manager
fig, ax = plt.subplots(figsize=(12, 5))

top_conv = mgr_active.sort_values('%_leads', ascending=False)

sns.barplot(data=top_conv, x='%_leads', y='deal_owner_name', color=colors['accent'], ax=ax)

# Value labels
ax.bar_label(ax.containers[0], fmt='%.1f', padding=3)

# Titles and axis labels
ax.set_title('Processed Deals by Manager')
ax.set_xlabel('Processed Deals (%)')
ax.set_ylabel('')

fig.tight_layout()

plt.show()

### Lead-Volume Insights

- **Charlie Davis** handles the largest share of the team's observed leads.
- The top five managers account for more than half of the observed lead volume, indicating a concentrated workload.
- Some managers show high call activity but relatively low deal ownership, which may reflect different responsibilities within the sales process.

## 4. Revenue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Realized revenue proxy (initial_amount_paid)
top_rev = mgr_active.sort_values('revenue', ascending=False)
sns.barplot(data=top_rev, x='revenue', y='deal_owner_name', color=colors['accent'], ax=axes[0])
axes[0].bar_label(axes[0].containers[0], fmt='%.0f', padding=3)
axes[0].set_title('Revenue (Actual) by Manager')
axes[0].set_xlabel('Revenue (€)')
axes[0].set_ylabel('')

# 2. Potential contract value (offer_total_amount)
top_rev_pot = mgr_active.sort_values('revenue_potential', ascending=False)
sns.barplot(data=top_rev_pot, x='revenue_potential', y='deal_owner_name', color=colors['accent'], ax=axes[1])
axes[1].bar_label(axes[1].containers[0], fmt='%.0f', padding=3)
axes[1].set_title('Revenue (Potential) by Manager')
axes[1].set_xlabel('Revenue Potential (€)')
axes[1].set_ylabel('')

plt.suptitle('Actual vs Potential Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Revenue Insights

The analysis compares two complementary measures:

- **Realized revenue proxy:** `initial_amount_paid`, representing money already recorded as paid.
- **Potential revenue:** `offer_total_amount`, representing the full contract value if the customer completes the course.

Managers should therefore be compared using both revenue measures together with conversion and lead volume rather than by a single ranking.

## 5. Call Performance

In [ ]:
calls_mgr = calls.groupby('call_owner_name').agg(total_calls=('id', 'count'), successful_calls=('is_successful', 'sum')).reset_index()

calls_mgr['success_rate'] = (calls_mgr['successful_calls'] / calls_mgr['total_calls'] * 100).round(1)
calls_mgr = calls_mgr.sort_values('total_calls', ascending=False)
calls_mgr

In [ ]:
# Minimum call-volume threshold to reduce noise from very small samples
calls_mgr = calls_mgr[calls_mgr['total_calls'] >= 500]
print(f'Managers after call-volume filtering: {len(calls_mgr)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_calls = calls_mgr.sort_values('total_calls', ascending=False)
sns.barplot(data=top_calls, x='total_calls', y='call_owner_name', color=colors['accent'], ax=axes[0])
axes[0].set_title('Total Calls by Manager')
axes[0].set_xlabel('Total Calls')
axes[0].set_ylabel('')

top_rate = calls_mgr.sort_values('success_rate', ascending=False)
sns.barplot(data=top_rate, x='success_rate', y='call_owner_name', color=colors['accent'], ax=axes[1])
axes[1].set_title('Success Rate by Manager (%)')
axes[1].set_xlabel('Success Rate (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### Call Performance Insights

- Call workload is concentrated among a subset of managers.
- **Derek James** and **Viktor Barnes** have particularly high successful-call rates in the analyzed data.
- Some high-volume callers do not own Deals, which supports the possibility of specialized call-center roles.
- Call success should not be treated as a direct measure of sales conversion because the two activities can belong to different stages or roles in the CRM process.

## 6. Lost-Deal Reasons

In [ ]:
reasons = deals['lost_reason'].value_counts(normalize=True) *100
reasons = reasons.round(2).sort_values(ascending=False)

In [ ]:
top_10_reasons = reasons.head(10)

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(x=top_10_reasons.values, y=top_10_reasons.index, color=colors['accent'], ax=ax)

# Value labels
ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)

# Titles and axis labels
ax.set_xlabel('Share of Lost Deals (%)')
ax.set_ylabel('')
ax.set_title('Distribution of Lost Reasons')

fig.tight_layout()

plt.show()

### Lost-Deal Insights

- **Doesn't Answer** is the most common recorded lost reason.
- A large share of losses is related to unsuccessful customer contact (`Doesn't Answer`, `Stopped Answering`, `Invalid Number`).
- `Non Target` and `Invalid Number` may indicate lead-quality or targeting issues.
- `Changed Decision` suggests a separate opportunity to investigate why initially interested leads abandon the purchase.
- In this dataset, communication and lead quality appear to be more prominent loss drivers than price-related reasons.